# NLP Feature Extraction
- Part-of-Speech Tagging
- Term Frequency-Inverse Document Frequency
- Sentiment Analysis
- Pretrained Word Embeddings
- Custom Word Embbeding Model
- Speaker / Speaker history

## Import Dependencies

In [1]:
import spacy
import pandas as pd
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from typing import List
from utils import load_model_and_tokenizer, load_inference_data
from sentiment_analysis_eval import get_predictions
import gzip
from gensim.models import Word2Vec
import re

# Helper Functions


Before extracting NLP features, we need functions to load the dataset in a flexible way.  
This section defines two helper functions:

1. **`load_df(filepath: str) -> pd.DataFrame`**  
   - Loads a dataset into a Pandas DataFrame from a given file path.  
   - Supports `.csv` and `.xlsx` formats.  
   - If an unsupported file type is provided, a `ValueError` is raised with a clear message.  
   - Includes error handling for missing files (`FileNotFoundError`).

2. **`load_sentences(filepath: str, text_column: str = "Sentence") -> pd.Series`**  
   - Uses `load_df()` to load the dataset.  
   - Returns only the column containing sentences (default: `"Sentence"`).  
   - This makes it easy to work directly with text data for NLP feature extraction. 

In [2]:
def load_df(filepath: str) -> pd.DataFrame:
    """Function to load a pd.DataFrame from a given filepath"""
    try:
        if filepath.endswith('.csv'):
            df = pd.read_csv(filepath)
        elif filepath.endswith('.xlsx'):
            df = pd.read_excel(filepath)
        else:
            raise ValueError("Unsupported file format. Please use .csv or .xlsx")
    except FileNotFoundError:
        raise FileNotFoundError(f"File not found: {filepath}")
    return df

def load_sentences(filepath: str, text_column: str = "Sentence") -> pd.Series:
    """Function to load a column from a pd.DataFrame into a list
    Author: Nick Belterman"""
    df = load_df(filepath)
    return df[text_column]


## Create Variables

1. **Language Code Mapping**  
   - A dictionary `language_code_mapping` maps ISO language codes to their full names.  
   - This mapping ensures flexibility when working with multilingual datasets, allowing easy switching between languages.

2. **Configurable Parameters**  
   - `language`: sets the desired language for analysis (`'nl'` for Dutch in this example).  
   - `transcription_file`: specifies the input transcription dataset (CSV format).  
   - `output_file`: defines the location where extracted NLP features will be saved.

3. **Spacy Model Download Reminder**  
   - A print statement reminds the user to download the appropriate spaCy language model using:  
     ```bash
     poetry run python -m spacy download {language}_core_news_lg
     ```  
   - This ensures that the correct model for tokenization, POS tagging, and embeddings is available before feature extraction begins.


In [3]:
language_code_mapping = {
    'nl': 'Dutch',
    'en': 'English',
    'de': 'German',
    'fr': 'French',
    'es': 'Spanish',
    'it': 'Italian',
    'pt': 'Portuguese',
    'ru': 'Russian',
    'zh': 'Chinese',
    'ja': 'Japanese',
    'ko': 'Korean',
    'ar': 'Arabic',
    'hi': 'Hindi',
    'bn': 'Bengali',
    'ur': 'Urdu',
    'vi': 'Vietnamese',
    'tr': 'Turkish',
    'sv': 'Swedish',
    'da': 'Danish',
    'no': 'Norwegian',
    'fi': 'Finnish',
    'pl': 'Polish',
    'cs': 'Czech',
    'hu': 'Hungarian',
    'ro': 'Romanian',
    'el': 'Greek',
    'he': 'Hebrew',
}

language = 'nl'  # Change this to the desired language code
transcription_file = "..\\data\\transcription\\csv\\transcription.csv"  # Change this to your transcription file path
output_file = '..\\data\\features\\NLP_features_output.csv'  # Change this to your desired output file path
print(f"Download model using `poetry run python -m spacy download {language}_core_news_lg` if not already installed.")

Download model using `poetry run python -m spacy download nl_core_news_lg` if not already installed.


## Function to Load Spacy Model

#### **Function: `load_spacy_model`**
- **Inputs:**
  - `language`: ISO language code specifying which spaCy model to load.  
  - `text_column` (default `'Sentence'`): The column in the dataset that contains text data.  

- **Process:**
  1. Constructs the spaCy model name dynamically: `"{language}_core_news_lg"`.  
     - Example: for Dutch (`nl`), it loads `nl_core_news_lg`.  
  2. Looks up the full language name from `language_code_mapping`.  
  3. Creates a configuration dictionary (`CONFIG`) that stores:  
     - The spaCy model name  
     - The human-readable language name  
     - The text column name  

- **Outputs:**
  - Returns a tuple:  
    1. The loaded spaCy language model.  
    2. The `CONFIG` dictionary for reference.  

- **Error Handling:**
  - If an unsupported language code is provided, a `ValueError` is raised with a clear message.


In [4]:
def load_spacy_model(
    language: str = 'nl', 
    text_column: str = 'Sentence',
    ) -> spacy.language.Language:
    """Load the spaCy model for the specified language.
    Args:
        language (str): Language code (default is 'nl' for Dutch).
    Returns:
        spacy.language.Language: Loaded spaCy language model.
    Raises:
        ValueError: If the language code is not supported.
    Authors: Floris Lokhorst, Nick Belterman
    """
    model_name = f"{language}_core_news_lg"
    try:
        language_name = language_code_mapping.get(language)
    except KeyError:
        raise ValueError(f"Unsupported language code: {language}")

    # Configuration
    CONFIG = {
        'spacy_model': model_name,
        'language': language_name,
        'text_column': text_column,  
    }

    return spacy.load(CONFIG['spacy_model']), CONFIG

## Functions to Extract POS tags

#### **Function: `extract_save_pos_tags`**
- **Inputs:**
  - `input_filepath`: Path to the dataset (CSV or Excel).  
  - `nlp_model`: The spaCy language model returned by `load_spacy_model`.  
  - `output_filepath`: Destination file for saving extracted features (default: `..\\data\\features\\NLP_features_output.csv`).  
  - `config`: Optional dictionary containing configuration settings (e.g., the text column name).  

- **Process:**
  1. Loads the dataset with `load_df`.  
  2. Determines the text column name (default: `'Sentence'`).  
  3. Ensures the output directory exists (creates it if necessary).  
  4. Loads sentences from the dataset using `load_sentences`.  
  5. Applies the spaCy pipeline to each sentence:
     - Each token in the sentence is tagged with its POS label (`token.text_POS`).  
     - The POS-tagged sentence is stored in a new DataFrame column `POS_tags`.  
  6. Adds placeholder columns `start_time` and `end_time` filled with `NaN` (likely for future alignment with temporal data).  

- **Output:**
  - A new CSV file containing the original dataset with an additional `POS_tags` column.  
  - A console message confirming the number of processed sentences and the output location.  

- **Error Handling:**
  - If saving fails, the exception is caught and a descriptive error message is printed.  

In [5]:
def extract_save_pos_tags(
    input_filepath: str, 
    nlp_model: spacy.language.Language, \
    output_filepath: str = '..\\data\\features\\NLP_features_output.csv',
    config: dict = None
) -> None:
    """Extract POS tags from text.
    
    Authors: Floris Lokhorst, Nick Belterman
    """
    df  = load_df(input_filepath)
    text_column = config.get('text_column', 'Sentence') if config else 'Sentence'
    if not os.path.exists(os.path.dirname(output_filepath)):
        os.makedirs(os.path.dirname(output_filepath))
    data = load_sentences(input_filepath, text_column=text_column)
    # Process each sentence and store the POS tags
    df['POS_tags'] = data.astype(str).apply(
        lambda text: ' '.join(f"{token.pos_}" for token in nlp_model(text.strip())) if text and text.strip() else ''
    )
    

    # Save the DataFrame to a CSV file
    try:
        df.to_csv(output_filepath, index=False)
        print(f"Successfully processed {len(data)} sentences and saved the output to {output_filepath}")
    except Exception as e:
        print(f"Error saving the CSV file: {e}")

# Code to extract and save POS tags using spacy

In [6]:
nlp, config = load_spacy_model()
extract_save_pos_tags(transcription_file, nlp, output_file, config)


Successfully processed 46 sentences and saved the output to ..\data\features\NLP_features_output.csv


# TF-IDF

# Function to load and fit TF-IDF Vectorizer on our chosen episode 

#### **Function: `load_fit_tfidf`**
- **Inputs:**
  - `sentences` (`List[str]`): A list of text sentences to process.

- **Process:**
  1. Initializes a `TfidfVectorizer` with the following settings:
     - `max_features=2000`: Limits the vocabulary to the 2,000 most important terms.  
     - `min_df=1`: Includes terms that appear in at least 1 document.  
     - `max_df=1.0`: Includes terms that appear in up to 100% of documents (all terms).  
     - `lowercase=True`: Converts all text to lowercase before tokenization.  
     - `stop_words=None`: No stopword removal is applied (all words are considered).  
     - `token_pattern=r'\b[a-záéíóúàèìòùâêîôûäëïöüç]+\b'`: Only alphabetic tokens (including accented characters) are considered.  
  2. Fits the TF-IDF vectorizer on the sentences.  
  3. Retrieves the vocabulary (list of terms included in the vectorizer).  

- **Outputs:**
  - `tfidf_matrix`: Sparse matrix where each row represents a sentence and each column represents a term, with TF-IDF weights.  
  - `vocab`: List of feature terms extracted by the vectorizer.  

- **Console Messages:**
  - Prints when TF-IDF fitting starts.  
  - Prints the total size of the vocabulary.  

In [7]:
def load_fit_tfidf(
    sentences: List[str], 
) -> (np.ndarray, List[str]):
    """Load and fit a TF-IDF vectorizer on the provided sentences.
    
    Args:
        sentences (List[str]): List of sentences to fit the TF-IDF vectorizer.
    
    Authors: Jeroen Brower, Nick Belterman
    """
    vectorizer = TfidfVectorizer(
        max_features=2000,         
        min_df=1,                    
        max_df=1.0,                  
        lowercase=True,
        stop_words=None,             
        token_pattern=r'\b[a-záéíóúàèìòùâêîôûäëïöüç]+\b',  
    )
    print("Fitting TF-IDF...")
    tfidf_matrix = vectorizer.fit_transform(sentences)
    vocab = vectorizer.get_feature_names_out()
    print(f"Total vocabulary: {len(vocab)}")
    return tfidf_matrix, vocab

# Function to add the TF-IDF matrix to our feature extraction output file

#### Function: `create_tfidf_dataframe`
- **Inputs:**
  - `tfidf_matrix` (`np.ndarray`): Sparse TF-IDF matrix returned by `load_fit_tfidf`.  
  - `vocab` (`List[str]`): List of vocabulary terms corresponding to the matrix columns.  
  - `output_filepath` (default: `..\\data\\features\\NLP_features_output.csv`): Path to save the resulting CSV.  
  - `text_column` (default: `'Sentence'`): Name of the text column in the original dataset.  
  - `readable` (default `False`): If `True`, formats the TF-IDF features in a human-readable format showing top terms per sentence; otherwise, keeps all non-zero TF-IDF scores.

- **Process:**
  1. Loads the original transcription dataset and sentences.  
  2. For each sentence:
     - Converts its TF-IDF vector to an array.  
     - Extracts non-zero TF-IDF scores and maps them to words from `vocab`.  
     - If `readable=True`, selects top 8 words with the highest TF-IDF scores and formats as `"word(score)"`.  
     - If `readable=False`, includes all non-zero scores for completeness.  
     - Stores the formatted string in a new column `TF_IDF`.  
  3. Removes rows with missing text in the specified column.  
  4. Saves the updated DataFrame with TF-IDF features to the specified CSV file.  

- **Outputs:**
  - The CSV file now includes a `TF_IDF` column containing either readable or complete TF-IDF representations for each sentence.  
  - Console messages confirm the format used and the save location.


In [ ]:
def create_tfidf_dataframe(
    tfidf_matrix: np.ndarray, 
    vocab: List[str],
    input_filepath: str = '..\\data\\transcription\\csv\\transcription.csv',
    output_filepath: str = '..\\data\\features\\NLP_features_output.csv',
    text_column: str = 'Sentence',
) -> pd.DataFrame:
    """Create a DataFrame from the TF-IDF matrix and vocabulary.
    
    Args:
        tfidf_matrix (np.ndarray): The TF-IDF matrix.
        vocab (List[str]): The vocabulary list.
    
    Returns:
        pd.DataFrame: DataFrame with TF-IDF features.
    
    Authors: Jeroen Brower, Nick Belterman
    """
    df = load_df(transcription_file)
    sentences = load_sentences(transcription_file, text_column=text_column)

    # Convert to numerical lists format - only values > 0 in sentence order
    tfidf_vectors = []
    for i in range(len(sentences)):
        sentence = sentences[i]
        vector = tfidf_matrix[i].toarray()[0]
        sentence_words = []
        for word in sentence.lower().split():
            clean_word = re.sub(r'[^\wa-záéíóúàèìòùâêîôûäëïöüç]', '', word)
            if clean_word and clean_word in vocab:
                sentence_words.append(clean_word)
    
        sentence_order_values = []
        for word in sentence_words:
            if word in vocab:
                word_index = list(vocab).index(word)
                tfidf_score = vector[word_index]
                if tfidf_score > 0:
                    sentence_order_values.append(round(float(tfidf_score), 3))
    
        tfidf_vectors.append(sentence_order_values)
    
    # Add to dataframe
    df['TF_IDF'] = tfidf_vectors
    


    # Dropna
    df.dropna(subset=[text_column], inplace=True)
    # Add to dataframe
    
    df.to_csv(output_filepath, index=False)
    print(f"Saved TF-IDF features to {output_filepath}")

# Code to extract and save TF-IDF terms using Scikit-learn

In [ ]:
sentences = load_sentences(output_file, text_column='Sentence').dropna()
tfidf_matrix, vocab = load_fit_tfidf(sentences)
create_tfidf_dataframe(
    tfidf_matrix, 
    vocab, 
    input_filepath=output_file, 
    output_filepath=output_file, 
    text_column='Sentence'
)

Fitting TF-IDF...
Total vocabulary: 138
Saved TF-IDF features to ..\data\features\NLP_features_output.csv


In [10]:
df = pd.read_csv(output_file)
print(df['TF_IDF'][0])

[0.537, 0.329, 0.537, 0.375, 0.418]


# Sentiment Analysis

Model can be found in [Models_nlp_2526](https://edubuas-my.sharepoint.com/:f:/g/personal/225538_buas_nl/EuvOvnKbcLdPnsq8LGqDbJ0BP0DLEER7PMAwyS1bOy4JZg?e=olbDPM) and should be saved in `nlp_cia\\models\\` 

#### **Process Overview**
1. **Remove Missing Sentences**
   - Any rows in the DataFrame without a value in the `Sentence` column are dropped using `dropna(subset=['Sentence'])`.  
   - Ensures that only valid text is passed to the model.

2. **Load Model and Tokenizer**
   - `load_model_and_tokenizer(model_dir=...)` loads a pretrained model and its tokenizer from the specified directory.  
   - The tokenizer is used to convert sentences into token IDs suitable for model input.

3. **Prepare Sentences for Inference**
   - `load_inference_data(tokenizer, inference_data=df)` tokenizes the sentences in the DataFrame for model prediction.  
   - Returns tokenized `sentences` and an updated DataFrame.

4. **Generate Predictions**
   - `get_predictions(model, sentences)` runs the tokenized sentences through the model to obtain predictions.  
   - Predictions are printed to the console for verification.

5. **Save Predictions**
   - The updated DataFrame, now including model predictions, is saved to `output_file`.  

In [11]:
df = df.dropna(subset=['Sentence'])
# subset = df.iloc[:15]
tokenizer, model = load_model_and_tokenizer(model_dir=r"..\\models\\Model_14e03c00")
sentences, df = load_inference_data(tokenizer, inference_data=df)
preds = get_predictions(model, sentences)
print(f"Predictions: {preds}")
df.to_csv(output_file, index=False)

Loading model and tokenizer from: ..\\models\\Model_14e03c00
c:\Users\nickb\OneDrive\Documenten\GitHub\fae2-nlpr-group-group-4-1\nlp_cia\src
texts = ['Ah, de ouders van Maartje', 'Ja, hallo', 'Hallo', 'Zo', 'Nou', 'Ja, wat fijn dat jullie allebei naar dit tienminutengesprek konden komen', 'Waar zou ik beginnen', 'Is er iets', 'Heeft ze zich misdragen', 'Zijn de cijfers onvoldoende', 'Nou, Maartje tekent nogal graag, h', 'O, ja, ja, ja', 'Ze is een heel creatief meisje', 'Ja, dat heeft ze van mij', 'Maar is het te veel', 'Moet ze minder tekenen', 'Want dan ga ik dat tegen haar zeggen', 'Ja, ik was vroeger ook altijd bij potten', 'Gewoon, hij stift in de weer de hele dag', 'Nou, het gaat ons meer om wat ze tekent', 'Laatst was de opdracht teken je lievelingsdier', 'En dan komt Maartje met dit', 'Tja, het is op zich wel goed getekend', 'Ja, Miepie is ook goed geschreven', 'Vorige week was de opdracht teken een zelfportret', 'Daar ziet u wat voorbeelden van de andere kinderen', 'Leuk', 'Ja

## Custom Word Embeddings
### We used the dutch OpenSubtitles Corpus as it matches our goal for TV emotion classification

OpenSubtitles Dutch Corpus (OPUS Collection)
Source: Movie and TV subtitle translations from OpenSubtitles.org, compiled by the OPUS project NlplPapers with Code

https://opus.nlpl.eu/OpenSubtitles/nl&en/v2024/OpenSubtitles

https://www.opensubtitles.org/en/search/subs


Scale: Part of collection spanning 60 corpora in 58 languages with 2.6 billion sentences total OpenSubtitles parallel corpora

Domain: Movies and television shows (conversational dialogue)

Language variety:  Dutch

Text type: Subtitle transcripts representing natural dialogue and speech patterns

Countries of origin: Dutch-language films and international content dubbed/subtitled in Dutch

Temporal range: Various decades of film/TV content up to recent releases

Data structure: Monolingual Dutch sentences extracted from subtitle files



## Code to extract relevant information from OpenSubtitles dataset

- **Process:**
  1. **Decompress the File**
     - Reads the `.gz` file using `gzip.open`.  
     - Writes the uncompressed content to a new file (`.gz` suffix removed).  

  2. **Read Lines and Prepare Sentences**
     - Iterates through each line of the uncompressed file.  
     - Strips whitespace and splits the line into words.  
     - Only lines with at least one word are added to the `sentences` list.  
     - The length of each line (number of words) is recorded in `line_lengths`.  
     - Stops processing once `max_sentences` is reached.

  3. **Preview and Statistics**
     - Prints the first 10 lines of the corpus for inspection.  
     - Computes and prints summary statistics:  
       - Total lines processed  
       - Number of sentences selected for training  
       - Average tokens per sentence  
       - Minimum and maximum tokens per sentence  

- **Outputs:**
  - `sentences`: List of tokenized sentences (each sentence is a list of words).  
  - `output_file`: Path to the uncompressed text file.


In [12]:
def extract_and_prepare_corpus(gz_file_path, max_sentences=1_000_000):
    """Extract corpus, show statistics, and prepare for training"""
    
    # Extract the file
    output_file = gz_file_path.replace('.gz', '')
    
    with gzip.open(gz_file_path, 'rt', encoding='utf-8') as f_in:
        with open(output_file, 'w', encoding='utf-8') as f_out:
            f_out.write(f_in.read())
    
    sentences = []
    line_lengths = []
    
    print("First 10 lines:")
    with open(output_file, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f):
            line = line.strip()
            
            # Show first 10 lines
            if line_num < 10:
                print(f"{line_num+1}: {line}")
            
            if line:
                words = line.split()
                if len(words) >= 1:
                    sentences.append(words)
                    line_lengths.append(len(words))
                
                # Stop at max for training
                if len(sentences) >= max_sentences:
                    break
    
    # Show statistics
    total_processed = line_num + 1
    print(f"\nDataset Statistics:")
    print(f"Total lines processed: {total_processed:,}")
    print(f"Sentences for training: {len(sentences):,}")
    print(f"Average tokens per sentence: {sum(line_lengths)/len(line_lengths):.1f}")
    print(f"Min/Max tokens: {min(line_lengths)}/{max(line_lengths)}")
    
    return sentences, output_file

In [13]:
training_sentences, corpus_file = extract_and_prepare_corpus("nl.tok.gz")

FileNotFoundError: [Errno 2] No such file or directory: 'nl.tok.gz'

#### **Function: `train_and_evaluate_model`**
- **Inputs:**
  - `sentences`: List of tokenized sentences (from `extract_and_prepare_corpus`).  
  - `params`: Dictionary of Word2Vec hyperparameters, such as `vector_size`, `window`, `min_count`, `sg`, `epochs`, and `workers`.  
  - `name`: Name of the model/configuration for identification.

- **Process:**
  1. Prints the model name and hyperparameters for clarity.  
  2. Trains a Word2Vec model using the `gensim` library:  
     - `vector_size`: Dimensionality of embeddings.  
     - `window`: Context window size.  
     - `min_count`: Ignores words with frequency below this threshold.  
     - `sg`: 1 = Skip-gram, 0 = CBOW.  
     - `epochs`: Number of training iterations.  
     - `workers`: Number of CPU threads for parallel training.  
  3. **Quick Evaluation:**  
     - Prints the vocabulary size.  
     - Checks similarity for example words (`'goed'`, `'slecht'`, `'mooi'`, `'blij'`).  
     - Prints the top 3 most similar words for each example.

- **Outputs:**
  - Returns the trained Word2Vec model instance.

#### **Testing Multiple Configurations**
- Several configurations (`configs`) are defined to test different hyperparameters:
  - `Baseline`: Standard settings.  
  - `larger_vector_size`: 500-dimensional embeddings.  
  - `smaller_window`: Reduced context window.  
  - `higher_mincount`: Ignores rarer words.  
  - `cbow_model`: Uses CBOW instead of Skip-gram.  
  - `Higher_epochs`: Trains for more epochs.  
- Each configuration is trained and stored in the `models` dictionary.


In [ ]:
def train_and_evaluate_model(sentences, params, name):
    """Train model with specific parameters and evaluate"""
    print(f"\n=== Training {name} ===")
    print(f"Parameters: {params}")
    
    model = Word2Vec(sentences=sentences, **params)
    
    # Quick evaluation
    test_words = ['goed', 'slecht', 'mooi', 'blij']
    print(f"Vocabulary size: {len(model.wv.index_to_key)}")
    
    for word in test_words:
        if word in model.wv:
            similar = model.wv.most_similar(word, topn=3)
            print(f"'{word}': {[w for w, score in similar]}")
    
    return model

# Test different configurations
configs = {
    "Baseline": {"vector_size": 200, "window": 5, "min_count": 5, "sg": 1, "epochs": 10, "workers": 8},
    "larger_vector_size": {"vector_size": 500, "window": 5, "min_count": 5, "sg": 1, "epochs": 10, "workers": 8},
    "smaller_window": {"vector_size": 200, "window": 3, "min_count": 5, "sg": 1, "epochs": 10, "workers": 8},
    "higher_mincount": {"vector_size": 200, "window": 5, "min_count": 10, "sg": 1, "epochs": 10, "workers": 8},
    "cbow_model": {"vector_size": 200, "window": 5, "min_count": 5, "sg": 0, "epochs": 10, "workers": 8},
    "Higher_epochs": {"vector_size": 200, "window": 5, "min_count": 5, "sg": 0, "epochs": 20, "workers": 8}
}

models = {}
for name, params in configs.items():
    models[name] = train_and_evaluate_model(training_sentences, params, name)

NameError: name 'training_sentences' is not defined

Parameter Justification:
- vector_size = 500: Balance between expressiveness and efficiency and no longer had the goed/slecht pairing
- window = 5: Captures sufficient conversational context
- min_count = 5: Filters noise, reduces vocab from 52K to 30K words
- sg = 1 (Skip-gram): Better for rare emotional vocabulary than CBOW
- epochs = 10: Sufficient convergence without overfitting

In [ ]:
print(f"Using {len(training_sentences)} sentences for training")
model = Word2Vec(
    sentences=training_sentences,
    vector_size=500,
    window=5,
    min_count=5,
    workers=4,
    epochs=10,
    sg=1,
    seed=42
)

model.save("dutch_custom_embeddings_final.model")

print(f"Training completed with {len(model.wv.index_to_key)} vocabulary words")

NameError: name 'training_sentences' is not defined

### Observed Results:
- Successfully learned semantic clusters (e.g., "mooi" with "prachtig", "fraai")
- Identified challenge: antonyms co-occur in dialogue


## Pretrained Word Embeddings

#### **Steps and Explanation**

1. **Load Dataset**
   - Prints the current working directory for reference.  
   - Reads the CSV file `wie_is_de_mol_sentiment.csv` into a Pandas DataFrame `df`.  
   - The DataFrame is expected to contain a `sentence` column with text data.

2. **Load spaCy Model**
   - Loads the Dutch large language model: `nl_core_news_lg`.  
   - Determines the vector size (`vec_size`) of the embeddings provided by the model.

3. **Define Sentence Embedding Function**
   - `sent_embedding_spacy(text: str) -> np.ndarray`: Computes a fixed-length vector for a sentence.  
     - Converts the input text into a spaCy `Doc` object.  
     - Collects the vector of each token that has a pretrained vector.  
     - Averages token vectors to produce a single sentence-level embedding.  
     - If no tokens have vectors, returns a zero vector of length `vec_size`.

4. **Compute Sentence Embeddings**
   - Applies `sent_embedding_spacy` to each sentence in the DataFrame.  
   - Stores results in a new column `embedding`.  

5. **Stack Embeddings into Array**
   - Converts the column of individual embeddings into a single 2D NumPy array (`embeddings_array`).  
   - Prints the shape of the array: `(number of sentences, embedding dimension)`.


In [ ]:
print(os.getcwd())
df = pd.read_csv(r"../data/wie_is_de_mol_sentiment.csv")

nlp = spacy.load("nl_core_news_lg")
vec_size = nlp.vocab.vectors_length

# Function to compute sentence embeddings
def sent_embedding_spacy(text: str) -> np.ndarray:
    doc = nlp(text)
    vecs = [t.vector for t in doc if t.has_vector]
    return np.mean(vecs, axis=0) if vecs else np.zeros(vec_size, dtype=np.float32)

# Compute embeddings for each row in the 'text' column
df['embedding'] = df['sentence'].apply(sent_embedding_spacy)
embeddings_array = np.stack(df['embedding'].values)
print(embeddings_array.shape)

c:\Users\nickb\OneDrive\Documenten\GitHub\fae2-nlpr-group-group-4-1\nlp_cia\src
(1083, 300)
